In [1]:
import torch
from jaxtyping import Float

In [2]:
N_QUERIES = 32
N_KEYS = 32
BATCH = 4
Q_TILE_SIZE = 16
K_TILE_SIZE = 16
SCALE = 1


In [3]:
def _make_attn_fwd_inputs(device=None):
    torch.random.manual_seed(0)
    n_queries = N_QUERIES
    n_keys = N_KEYS
    batch_size = BATCH
    D = 64
    Q = torch.randn(batch_size, n_queries, D, device=device, dtype=torch.float64)
    K = torch.randn(batch_size, n_keys, D, device=device, dtype=torch.float64)
    V = torch.randn(batch_size, n_keys, D, device=device, dtype=torch.float64)
    O = torch.zeros(batch_size, n_queries, D, device=device, dtype=torch.float64)
    L = torch.zeros(batch_size, n_queries, device=Q.device, dtype=torch.float64)

    return Q, K, V, O, L


def _make_attn_bwd_inputs(device=None):
    torch.random.manual_seed(0)
    n_queries = N_QUERIES
    n_keys = N_KEYS
    batch_size = BATCH
    D = 64
    dQ = torch.zeros(batch_size, n_queries, D, device=device, dtype=torch.float64)
    dK = torch.zeros(batch_size, n_keys, D, device=device, dtype=torch.float64)
    dV = torch.zeros(batch_size, n_keys, D, device=device, dtype=torch.float64)
    dO = torch.randn(batch_size, n_queries, D, device=device, dtype=torch.float64)

    return dQ, dK, dV, dO

In [4]:
Q, K, V, O, L = _make_attn_fwd_inputs()
dQ, dK, dV, dO = _make_attn_bwd_inputs()


In [5]:
@torch.no_grad
def flash_attn_fwd(
    Q: Float[torch.Tensor, "b queries d_q"],
    K: Float[torch.Tensor, "b keys d_q"],
    V: Float[torch.Tensor, "b keys d_v"],
    O_out: Float[torch.Tensor, "b queries d_v"],
    L_out: Float[torch.Tensor, "b queries"],
    N_QUERIES: int,
    N_KEYS: int,
    Q_TILE_SIZE: int,
    K_TILE_SIZE: int,
    scale: float,
):
    Q_TILES = (N_QUERIES + Q_TILE_SIZE - 1) // Q_TILE_SIZE
    K_TILES = (N_KEYS + K_TILE_SIZE - 1) // K_TILE_SIZE

    B = Q.size(0)

    for i in range(Q_TILES):  # Q outer loop
        Q_tile = Q[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]  # HBM -> SRAM
        O_tile = O_out[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]  # HBM -> SRAM

        prev_m = torch.zeros((B, Q_TILE_SIZE), device=Q.device)
        l = torch.zeros((B, Q_TILE_SIZE), device=Q.device)

        for j in range(K_TILES):  # K V inner loop
            # compute P
            K_tile = K[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]  # HBM -> SRAM

            S_tile = Q_tile @ K_tile.transpose(-1, -2) / scale  # SMEM

            tile_m = S_tile.max(dim=-1).values
            m = torch.maximum(prev_m, tile_m)

            P_tile = torch.exp(S_tile - m.unsqueeze(-1))

            exp_correction = torch.exp(prev_m - m)
            prev_m = m

            l = exp_correction * l + P_tile.sum(dim=-1)

            # compute O
            V_tile = V[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]
            O_tile = O_tile * exp_correction.unsqueeze(-1) + P_tile @ V_tile

        O_tile = (1 / l.unsqueeze(-1)) * O_tile

        # write results
        O_out[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :] = O_tile  # SMEM -> HBM
        L_out[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1)] = prev_m + torch.log(l)  # SMEM -> HBM

flash_attn_fwd(Q, K, V, O, L, N_QUERIES, N_KEYS, Q_TILE_SIZE, K_TILE_SIZE, SCALE)

In [6]:
Q_ref = Q.detach()
K_ref = K.detach()
V_ref = V.detach()
Q_ref.requires_grad_()
K_ref.requires_grad_()
V_ref.requires_grad_()


def attn_fwd(Q, K, V, scale):
    return torch.softmax(Q @ K.transpose(-1, -2) / scale, dim=-1) @ V


O_ref = attn_fwd(Q_ref, K_ref, V_ref, SCALE)

dQ_ref, dK_ref, dV_ref = torch.autograd.grad(outputs=[O_ref], inputs=[Q_ref, K_ref, V_ref], grad_outputs=[dO])


In [7]:
assert torch.allclose(O, O_ref, atol=1e-5)

In [8]:
@torch.no_grad
def flash_attn_bwd(
    Q: Float[torch.Tensor, "b queries d_q"],
    dQ_out: Float[torch.Tensor, "b queries d_q"],
    K: Float[torch.Tensor, "b keys d_q"],
    dK_out: Float[torch.Tensor, "b keys d_q"],
    V: Float[torch.Tensor, "b keys d_v"],
    dV_out: Float[torch.Tensor, "b keys d_v"],
    O: Float[torch.Tensor, "b queries d_v"],
    dO: Float[torch.Tensor, "b queries d_v"],
    L: Float[torch.Tensor, "b queries"],
    N_QUERIES: int,
    N_KEYS: int,
    Q_TILE_SIZE: int,
    K_TILE_SIZE: int,
    scale: float,
):
    Q_TILES = (N_QUERIES + Q_TILE_SIZE - 1) // Q_TILE_SIZE
    K_TILES = (N_KEYS + K_TILE_SIZE - 1) // K_TILE_SIZE

    D = (O * dO).sum(dim=-1)

    for j in range(K_TILES):  # K V outer loop
        K_tile = K[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]  # HBM -> SRAM
        V_tile = V[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]  # HBM -> SRAM

        dK_out_tile = dK_out[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]
        dV_out_tile = dV_out[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]

        for i in range(Q_TILES):
            Q_tile = Q[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]  # HBM -> SRAM
            dO_tile = dO[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]  # HBM -> SRAM
            L_tile = L[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1)]  # HBM -> SRAM
            D_tile = D[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1)]  # HBM -> SRAM

            # check
            S_tile = Q_tile @ K_tile.transpose(-2, -1) / scale
            P_tile = torch.exp(S_tile - L_tile.unsqueeze(-1))

            # 1
            dV_out_tile = dV_out_tile + P_tile.transpose(-2, -1) @ dO_tile  # TARGET

            # 2
            dP_tile = dO_tile @ V_tile.transpose(-2, -1)

            # 3
            dS_tile = P_tile * (dP_tile - D_tile.unsqueeze(-1))

            # 5
            dK_out_tile = dK_out_tile + dS_tile.transpose(-2, -1) @ Q_tile / scale  # TARGET

        dV_out[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :] = dV_out_tile
        dK_out[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :] = dK_out_tile

    # для torch реализации это не нужно
    for i in range(Q_TILES):
        Q_tile = Q[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]  # HBM -> SRAM
        dO_tile = dO[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]  # HBM -> SRAM
        dQ_out_tile = dQ_out[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]
        L_tile = L[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1)]  # HBM -> SRAM
        D_tile = D[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1)]  # HBM -> SRAM

        for j in range(K_TILES):
            K_tile = K[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]  # HBM -> SRAM
            V_tile = V[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]  # HBM -> SRAM

            # check
            S_tile = Q_tile @ K_tile.transpose(-2, -1) / scale
            P_tile = torch.exp(S_tile - L_tile.unsqueeze(-1))

            # 2
            dP_tile = dO_tile @ V_tile.transpose(-2, -1)

            # 3
            dS_tile = P_tile * (dP_tile - D_tile.unsqueeze(-1))

            # 4
            dQ_out_tile = dQ_out_tile + dS_tile @ K_tile / scale  # TARGET

        dQ_out[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :] = dQ_out_tile


flash_attn_bwd(Q, dQ, K, dK, V, dV, O, dO, L, N_QUERIES, N_KEYS, Q_TILE_SIZE, K_TILE_SIZE, SCALE)


In [9]:
assert torch.allclose(dV, dV_ref, atol=1e-5)
assert torch.allclose(dK, dK_ref, atol=1e-5)
assert torch.allclose(dQ, dQ_ref, atol=1e-5)